In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

In [2]:
img_size = 28
batch_size = 128
epochs = 50
lr = 1e-4
weight_decay = 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

test_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(
    root="./data", train=True, download=True, transform=train_transform
)

test_dataset = datasets.MNIST(
    root="./data", train=False, download=True, transform=test_transform
)

trainloader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True
)

testloader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False
)

In [5]:
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.GroupNorm(16, 64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.1),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.GroupNorm(16, 128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.GroupNorm(16, 256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.3),

            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )

        self.fc = nn.Linear(256, 10)

    def forward(self, x):
        x = self.features(x)
        return self.fc(x)

net = Classifier().to(device)

total_params = sum(p.numel() for p in net.parameters())
print(f"Total Parameters: {total_params:,}")

Total Parameters: 373,130


In [6]:
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import argparse
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pre_epoch = 0
BATCH_SIZE = 128
LR = 0.001

EPOCH = 500
pre_epoch = 0

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    net.parameters(),
    lr=lr,
    weight_decay=weight_decay
)

In [7]:
best_acc = 0.0
best_model = None

for epoch in range(epochs):
    net.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(trainloader, desc=f"Epoch {epoch+1}/{epochs}")

    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = outputs.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()

        pbar.set_postfix({
            "Loss": f"{running_loss / len(pbar):.3f}",
            "Acc": f"{100. * correct / total:.2f}%"
        })

    if epoch % 10 == 0:
        torch.save(best_model, f"best_fashion_mnist_cnn_{epoch}.pth")

    # -------------------------
    # Evaluation
    # -------------------------
    net.eval()
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = net(inputs)
            _, preds = outputs.max(1)
            test_total += labels.size(0)
            test_correct += preds.eq(labels).sum().item()

    test_acc = 100. * test_correct / test_total
    print(f"--- Epoch {epoch+1} | Test Acc: {test_acc:.2f}% ---")

    if test_acc > best_acc:
        best_acc = test_acc
        best_model = net.state_dict()

Epoch 1/50:   0%|          | 0/469 [00:00<?, ?it/s]

--- Epoch 1 | Test Acc: 82.32% ---


Epoch 2/50:   0%|          | 0/469 [00:00<?, ?it/s]

--- Epoch 2 | Test Acc: 87.28% ---


Epoch 3/50:   0%|          | 0/469 [00:00<?, ?it/s]

--- Epoch 3 | Test Acc: 89.96% ---


... (90 outputs left out)

Epoch 49/50:   0%|          | 0/469 [00:00<?, ?it/s]

--- Epoch 49 | Test Acc: 98.61% ---


Epoch 50/50:   0%|          | 0/469 [00:00<?, ?it/s]

--- Epoch 50 | Test Acc: 98.62% ---


In [10]:
# torch.save(net.state_dict(), "mnist_cnn_wei.pth")